# GameTheory-13d : Optimistic Counterfactual Regret Minimization (OFTRL stable-prédictif)

**Navigation** : [<< 13c-Safe-Subgame-Solving-Csharp](GameTheory-13c-Safe-Subgame-Solving-CSharp.ipynb) | [Index](README.md) | [13-ImperfectInfo-CFR >>](GameTheory-13-ImperfectInfo-CFR-Python.ipynb)

> **Notebook compagnon** de `GameTheory-13-ImperfectInfo-CFR-Python.ipynb`. La fondation sequence form / treeplex / regret circuits du GT-13 est réutilisée ici, sans dupliquer de moteur CFR monolithique.

## Counterfactual Regret Minimization — variante optimiste

Ce notebook couvre la famille **optimiste** du CFR (OFTRL / Optimistic Follow-the-Regularized-Leader), telle que formalisée pour les jeux à information imparfaite par :

> Farina, Kroer, Brown, Sandholm, *Stable-Predictive Optimistic Counterfactual Regret Minimization*, **ICML 2019** (PMLR 97).

### Pourquoi un notebook compagnon (13d) plutôt qu'une section du 13 ?

- Le notebook GT-13 principal est gelé sur l'attendu CFR/CFR+/MCCFR/OpenSpiel ; ce compagnon isole le scope OFTRL.
- Le scope OFTRL est ciblé, plusieurs pages, et la comparaison contrôlée à 4 solveurs (CFR / CFR+ / DCFR / OFTRL) demande une visualisation stable.
- Le présent notebook réutilise le socle du GT-13 et n'écrit qu'**un seul** nouveau solveur (`OFTRLSolver`).

### Objectifs d'apprentissage

- Comprendre la **garantie théorique** $\mathcal{O}(T^{-3/4})$ de l'OFTRL stable-prédictif vs $\mathcal{O}(T^{-1/2})$ du CFR/CFR+.
- Distinguer **mises à jour simultanées** (chaque joueur update avec les regrets de l'itération précédente) et **mises à jour alternées** (un joueur à la fois).
- Apprendre à régler empiriquement $(\kappa, \alpha, \beta)$ par profondeur de treeplex.
- Évaluer honnêtement : sur Kuhn poker, **DCFR reste empiriquement premier** sur 4 sous-jeux testés par les auteurs — l'OFTRL n'est pas une amélioration universelle.


## Préambule : socle #13350 réutilisé

Le présent notebook réutilise les classes suivantes du notebook principal `GameTheory-13-ImperfectInfo-CFR-Python.ipynb` (livré sur main par `#13350`, commit `05960df90`) :

- `KuhnPoker` (cellule 5)
- `RegretMatcher` (cellule 8)
- `CFRSolver` (cellule 11)
- `CFRPlusSolver(CFRSolver)` (cellule 20)
- `MCCFRSolver(CFRSolver)` (cellule 23, base pour DCFR dans la littérature)

Ces classes sont chargées par `importlib.util.spec_from_file_location` depuis le répertoire courant — **aucun moteur CFR monolithique tiers n'est recopié**. Si le notebook principal est renommé, ajuster `MAIN_NOTEBOOK` ci-dessous.


In [1]:
# Fix #17882: GameTheory-13d-Optimistic-CFR — le chargeur de socle exec les cellules
# cibles de GT-13 dans un namespace vide (les imports ne sont pas charges).
# Fix Output-flood ratchet : tqdm est substitue par un wrapper silencieux
# (_TqdmSilent) — sans lui, les cellules chargees en verbose=True (CFR/MCCFR)
# remplissent la cellule 2 du carnet companion de barres tqdm (mesure : 89
# streams vs 66 en base, depasse CELL_CAP=50). Allègement declare dans le
# body de la PR (#17882) — barres tqdm = sorties legitimes de transparence,
# mais neutres pour le ratchet.

# Imports top-level : visibles dans le globals du kernel pour les cellules
# suivantes (cellule 4 = OFTRLSolver utilise np, Dict, defaultdict).
import os
import re
import json
import numpy as np
from typing import List, Dict
from collections import defaultdict

# OpenSpiel + tqdm (optionnels mais references par les classes chargees).
try:
    import pyspiel
    OPENSPIEL_AVAILABLE = True
except ImportError:
    pyspiel = None
    OPENSPIEL_AVAILABLE = False


class _TqdmSilent:
    """tqdm no-op : iterateur silencieux compatible avec l'API tqdm.
    Substitue tqdm dans module_globals avant l'exec des cellules cibles
    du notebook GT-13 principal : les cellules en verbose=True (notamment
    la cellule d'entrainement MCCFRSolver.train(50000, verbose=True)) tournent sans
    ecrire de barres de progression. Sortie : meme comportement qu'un
    iterable, aucun caractere emis. Le progres reel reste mesurable via
    perf_counter si necessaire (cf. la cellule de benchmark 4 seeds)."""
    def __init__(self, *args, **kwargs):
        self._iter = kwargs.get('iterable', args[0] if args else None)

    def __iter__(self):
        return iter(self._iter) if self._iter is not None else iter(())

    def __next__(self):
        if self._iter is None:
            raise StopIteration
        return next(iter(self._iter))

    def update(self, n=1):
        pass

    def set_description(self, desc=None, refresh=True):
        pass

    def set_postfix(self, ordered_dict=None, refresh=True, **kwargs):
        pass

    def write(self, s, file=None, end="\n", nolock=False):
        pass

    def close(self):
        pass

    def refresh(self, nolock=False, lock_args=None):
        pass

    def clear(self, nolock=False):
        pass

    def reset(self, total=None):
        pass

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        return False


tqdm = _TqdmSilent


def _find_main_notebook() -> str:
    """Cherche le notebook GT-13 principal a proximite du notebook companion."""
    cwd = os.getcwd()
    candidates = [
        os.path.join(cwd, 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
        os.path.join(cwd, 'MyIA.AI.Notebooks', 'GameTheory', 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
        os.path.join(cwd, 'GameTheory', 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
        os.path.join(cwd, '..', 'GameTheory', 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
        os.path.join(cwd, '..', '..', 'GameTheory', 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
    ]
    for cand in candidates:
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(
        f"Main notebook GameTheory-13-ImperfectInfo-CFR-Python.ipynb introuvable. CWD={cwd!r}, candidates={candidates}"
    )


def _load_socle_from_notebook(path: str) -> dict:
    """Execute les cellules Python des classes cibles d'un notebook .ipynb
    et expose les classes dans un namespace isole pre-alimente. tqdm est
    injecte comme _TqdmSilent pour neutraliser les barres de progres des
    cellules cibles verbose=True."""
    module_globals = {
        '__name__': 'gt13_socle',
        'np': np,
        'List': List,
        'Dict': Dict,
        'defaultdict': defaultdict,
        'OPENSPIEL_AVAILABLE': OPENSPIEL_AVAILABLE,
        'tqdm': tqdm,
    }
    if pyspiel is not None:
        module_globals['pyspiel'] = pyspiel

    with open(path, 'r', encoding='utf-8') as fh:
        nb = json.load(fh)

    # Cibles localisees par CLASSE, jamais par index absolu : toute edition
    # markdown du carnet GT-13 (campagne densite #17040) decale les index et
    # faisait silencieusement disparaitre une classe du socle (KeyError
    # 'MCCFRSolver', golden-set 8/9 sur la PR #19842).
    target_classes = ['KuhnPoker', 'RegretMatcher', 'CFRSolver', 'CFRPlusSolver', 'MCCFRSolver']

    for cls in target_classes:
        src = None
        for cell in nb['cells']:
            if cell.get('cell_type') != 'code':
                continue
            lines = cell['source']
            cand = '\n'.join(lines) if isinstance(lines, list) else lines
            if re.search(rf'^class\s+{cls}\b', cand, re.M):
                src = cand
                break
        if src is None:
            raise RuntimeError(
                f"[socle] classe {cls} introuvable dans {os.path.basename(path)}"
            )
        try:
            exec(
                compile(src, f"<socle {os.path.basename(path)}:{cls}>", "exec"),
                module_globals,
            )
        except Exception as exc:  # noqa: BLE001
            raise RuntimeError(
                f"[socle] classe {cls} echec: {type(exc).__name__}: {exc}"
            ) from exc

    return module_globals


_MAIN_NB_PATH = _find_main_notebook()
socle = _load_socle_from_notebook(_MAIN_NB_PATH)
KuhnPoker = socle['KuhnPoker']
RegretMatcher = socle['RegretMatcher']
CFRSolver = socle['CFRSolver']
CFRPlusSolver = socle['CFRPlusSolver']
MCCFRSolver = socle['MCCFRSolver']
print(f"Socle charge depuis {os.path.basename(_MAIN_NB_PATH)}")
print(f"  Classes: KuhnPoker, RegretMatcher, CFRSolver, CFRPlusSolver, MCCFRSolver")
print(f"  OPENSPIEL_AVAILABLE: {OPENSPIEL_AVAILABLE}")
print(f"  CWD kernel: {os.path.basename(os.getcwd())}")
print(f"  tqdm substitue par _TqdmSilent (Fix Output-flood ratchet)")


Test Kuhn Poker:
  'pp' terminal? True
  'p' terminal? False
  Payoff 'bb' avec K vs J: 2
  Payoff 'bp' (fold): 1
  Info set J1 avec Q apres '': Q
Demo Regret Matching sur Pierre-Feuille-Ciseaux
Strategie moyenne apres 1000 iterations:
  Rock: 0.000, Paper: 0.999, Scissors: 0.000
  -> Converge vers Paper (meilleure reponse a Rock)
CFRSolver defini avec succes
Comparaison CFR vs CFR+

Entrainement CFR vanilla...


Entrainement CFR+...



Resultats apres 5000 iterations:


Optional module pokerkit_wrapper was not importable: No module named 'pokerkit'
  CFR:  utilite finale = 0.2450, exploitabilite = 0.002332
  CFR+: utilite finale = -0.0677, exploitabilite = 0.002881

Entrainement MCCFR...



MCCFR apres 50000 iterations:
  Utilite finale: -0.0555
Socle charge depuis GameTheory-13-ImperfectInfo-CFR-Python.ipynb
  Classes: KuhnPoker, RegretMatcher, CFRSolver, CFRPlusSolver, MCCFRSolver
  OPENSPIEL_AVAILABLE: True
  CWD kernel: GameTheory
  tqdm substitue par _TqdmSilent (Fix Output-flood ratchet)


## 1. Régime stable-prédictif — pourquoi l'optimisme aide en théorie

### 1.1 CFR vanilla comme Follow-the-Regularized-Leader

Le CFR est l'instance de **Follow-the-Regularized-Leader** avec :

$$
\text{regret}_i^T(I) = \max_{a \in A(I)} \sum_{t=1}^{T} \bigl( v_i^\sigma(I, a) - v_i^\sigma(I, \sigma_t(I)) \bigr)
$$

Avec une régularisation d'entropie, le CFR offre $\mathcal{O}(\sqrt{T})$ regret.

### 1.2 Le pas optimiste

L'**Optimistic FTRL** injecte une **prédiction** $m_{t+1}$ du regret à l'itération suivante :

$$
\sigma_{t+1} = \arg\min_{\sigma \in \Delta} \left\langle \eta \sum_{s=1}^{t} (\nabla \ell_s + m_s), \sigma \right\rangle
$$

Farina et al. (2019) proposent un pas optimiste **stable-prédictif** $(\kappa, \alpha, \beta)$ qui adapte la prédiction à la **profondeur du treeplex** du sous-jeu courant, garantissant :

$$
R_i^T \le \frac{\kappa}{\sqrt[4]{T}} + \frac{\alpha}{\sqrt{T}} + \beta \sqrt{\frac{\log T}{T}}
$$

### 1.3 Le bémol des auteurs

Sur Libratus et 4 grands sous-jeux, l'analyse empirique de Farina et al. montre que le pas théorique est **trop conservateur** sur les grands espaces : le **DCFR** (Discounted CFR) reste **premier**. Le présent notebook confronte cette attente à un banc mesuré : la section 2 calcule
l'**exploitabilité réelle** des quatre solveurs sur Kuhn poker et rapporte le classement
obtenu ici, qui n'est pas celui de l'étude citée. Distinguer « l'article l'affirme » de
« je le mesure sur mon jeu » est l'objet de la section 2.


In [2]:
# OFTRLSolver — le seul nouveau solveur de ce notebook
class OFTRLSolver(CFRSolver):
    """Counterfactual Regret Minimization avec pas optimiste stable-prédictif.

    Suit la formulation de Farina, Kroer, Brown, Sandholm (ICML 2019) section 3 :
    on injecte une prediction m_t+1 du regret a l'iteration suivante, calibree par
    profondeur de treeplex (kappa) avec amortissement (alpha, beta).

    Parametres :
        kappa   : amplitude de la prediction optimiste (~ sqrt(depth))
        alpha   : amortissement lineaire de la prediction par la profondeur
        beta    : regularisation tardive du pas (denominateur T + beta)
    """

    def __init__(self, kappa: float = 1.0, alpha: float = 0.5, beta: float = 1.0):
        super().__init__()
        # Memoire des predictions du pas optimiste (par info_set).
        self.predict_sum: Dict[str, np.ndarray] = defaultdict(
            lambda: np.zeros(self.game.NUM_ACTIONS)
        )
        self._kappa = kappa
        self._alpha = alpha
        self._beta = beta

    def get_strategy(self, info_set: str) -> np.ndarray:
        """Strategie courante = argmin sur (regret_sum + prediction)."""
        regrets = self.regret_sum[info_set] + self.predict_sum[info_set]
        positive_regrets = np.maximum(regrets, 0)
        normalizing_sum = positive_regrets.sum()
        if normalizing_sum > 0:
            return positive_regrets / normalizing_sum
        return np.ones(self.game.NUM_ACTIONS) / self.game.NUM_ACTIONS

    def cfr(self, history, cards, reach_probs, depth: int = 0):
        """Recursion CFR avec prediction optimiste injectee dans update."""
        # Cas terminal
        if self.game.is_terminal(history):
            payoff = self.game.get_payoff(history, cards)
            return np.array([payoff, -payoff])

        player = self.game.get_current_player(history)
        info_set = self.game.get_info_set(history, cards[player])
        strategy = self.get_strategy(info_set)
        action_utils = np.zeros(self.game.NUM_ACTIONS)
        node_value = 0.0

        for a in range(self.game.NUM_ACTIONS):
            new_history = history + ("p" if a == 0 else "b")
            child_value = self.cfr(new_history, cards,
                                   reach_probs * strategy[a],
                                   depth=depth + 1)
            action_utils[a] = child_value[player]
            node_value += strategy[a] * action_utils[a]

        # Regret du pas
        regrets = action_utils - node_value
        self.regret_sum[info_set] += reach_probs[1 - player] * regrets

        # PAS OPTIMISTE : la prediction m_{t+1} est une moyenne ponderee des
        # regrets positifs cumules. Le point proximal du regulariseur
        # quadratique ||m||^2 / 2 donne le pas canonique 1 / (T + 1) ; la
        # variante kappa / (T + beta) retenue ici en est une generalisation,
        # amortie par la profondeur via 1 / (1 + alpha * depth), pour que
        # (kappa, alpha, beta) soient trois leviers reellement actifs -- c'est
        # l'objet de l'exercice 1.
        # Mesure (Kuhn, info-set Qp) : le pas canonique et cette variante
        # donnent la meme trajectoire a 1e-3 pres -- [0.0320, 0.9680] contre
        # [0.0314, 0.9686] a t=10, valeurs identiques a t=500. La strategie
        # uniforme que l'ancien commentaire attribuait a cet amortissement
        # venait de l'info-set "Q", jamais visite (issue #21488) : elle
        # disparait aussi avec le pas canonique des que l'info-set correct est
        # lu, donc l'amortissement n'en etait pas la cause.
        T = max(self.iterations, 1)
        step = self._kappa / (T + self._beta)
        depth_factor = 1.0 / (1.0 + self._alpha * depth)
        self.predict_sum[info_set] += step * depth_factor * np.maximum(
            self.regret_sum[info_set], 0
        )
        # Accumulation de la strategie courante, ponderee par la probabilite
        # d'atteinte du joueur (la meme ligne que CFRSolver.cfr() dans le
        # notebook GT-13). Sans elle, strategy_sum reste nul pour cet info set
        # et get_average_strategy() renvoie l'uniforme. Correctif : PR #13604.
        self.strategy_sum[info_set] += reach_probs[player] * strategy
        return np.array([node_value, -node_value]) if player == 0 else np.array(
            [-node_value, node_value]
        )


# Demonstration rapide : 500 iterations Kuhn, CFR vs CFR+ vs OFTRL.
# Info-set affiche : "Qp" (joueur 1, carte Q, historique "p") -- un info-set
# effectivement visite par la recursion (le joueur 0 agit a la racine, donc
# l'info-set "Q" = carte Q + historique vide n'existe pas dans l'arbre parcouru
# avec cards=[0,1] ; l'afficher rendait les trois colonnes uniformes par
# construction). Voir issue #21488.
oftrl = OFTRLSolver(kappa=1.0, alpha=0.5, beta=1.0)
cfr = CFRSolver()
cfr_plus = CFRPlusSolver()
print(f"{'iter':>6} {'CFR avg':>10} {'CFR+ avg':>10} {'OFTRL avg':>10}")
for t in range(500):
    oftrl.iterations = t + 1
    oftrl.cfr("", [0, 1], np.array([1.0, 1.0]))
    cfr.cfr("", [0, 1], np.array([1.0, 1.0]))
    cfr_plus.cfr("", [0, 1], np.array([1.0, 1.0]))
    if t in (0, 9, 49, 99, 199, 299, 399, 499):
        oftrl_avg = oftrl.get_average_strategy("Qp").tolist()
        cfr_avg = cfr.get_average_strategy("Qp").tolist()
        cfr_plus_avg = cfr_plus.get_average_strategy("Qp").tolist()
        print(f"{t+1:>6} {str(cfr_avg):>20} {str(cfr_plus_avg):>20} {str(oftrl_avg):>20}")
print("(strategie 'Qp' = J1 carte Q face a un check -- info-set visite par la demo)")


  iter    CFR avg   CFR+ avg  OFTRL avg
     1           [0.5, 0.5]           [0.5, 0.5]           [0.5, 0.5]
    10         [0.05, 0.95]         [0.05, 0.95] [0.03138505996565611, 0.9686149400343439]
    50         [0.01, 0.99]         [0.01, 0.99] [0.005212071535540358, 0.9947879284644596]
   100       [0.005, 0.995]       [0.005, 0.995] [0.002551916874750216, 0.9974480831252498]
   200     [0.0025, 0.9975]     [0.0025, 0.9975] [0.001262845835560384, 0.9987371541644396]
   300 [0.0016666666666666668, 0.9983333333333333] [0.0016666666666666668, 0.9983333333333333] [0.0008390231029570693, 0.999160976897043]


   400   [0.00125, 0.99875]   [0.00125, 0.99875] [0.0006281950416979916, 0.999371804958302]


   500       [0.001, 0.999]       [0.001, 0.999] [0.0005020427381639588, 0.999497957261836]
(strategie 'Qp' = J1 carte Q face a un check -- info-set visite par la demo)


### 1.4 Le champion empirique : DCFR, la mémoire oublieuse

Le pas optimiste ci-dessus offre la meilleure **garantie théorique** ; le §1.3 annonçait que la première place **empirique** reste au DCFR — c'est la cellule suivante qui l'implémente, pour que la confrontation soit loyale. Son principe tient en une phrase : *les regrets anciens sont du bruit, oublions-les progressivement*. Concrètement, la cellule suivante emboîte trois mécanismes :

- un **facteur de regret** $\dfrac{t^{\alpha}}{t^{\alpha}+1}$ ($\alpha = 1{,}5$) appliqué à l'**incrément** de regret à chaque itération — ce facteur tend vers 1 quand $t$ grandit, il ne décroît donc **pas** la somme de regrets déjà accumulée ;
- un **facteur de stratégie** $\dfrac{t^{\beta}}{t^{\beta}+1}$ sur la contribution à la stratégie moyenne — désactivé par défaut ici ($\beta = 0$), il ne servirait qu'à ralentir l'uniformisation de la stratégie moyenne ;
- le **clipping CFR+** ($\gamma$) — les regrets négatifs sont écrêtés, la mise à jour ne propage que le regret positif.

Ce sont **ces trois lignes-là** que le code exécute, et c'est cette forme-là que le benchmark de la section 2 mesure. Une implémentation de référence décroît la somme accumulée au lieu de pondérer l'incrément : la différence n'est pas cosmétique, elle change ce qui est oublié et à quelle vitesse. Le lecteur qui compare ce carnet à l'article doit donc lire le code, pas seulement le nom du solveur.

L'ironie pédagogique est assumée : le solveur le plus simple conceptuellement — un CFR+ qui oublie — est celui que le benchmark suivant confrontera à l'OFTRL.


In [3]:
# DCFRDiscountedSolver — DCFR Discounted CFR (facteurs sur regrets et strategie)
# Forme EXECUTEE ici, celle que decrit le §1.4 : le facteur de regret vaut
# t**alpha / (t**alpha + 1) et porte sur l'INCREMENT de regret (il tend vers 1,
# la somme accumulee n'est pas decroissee) ; le facteur de strategie vaut
# t**beta / (t**beta + 1) et est neutre par defaut (beta = 0).
# Ce n'est donc pas la forme "decay de la somme accumulee" d'une implementation
# de reference : le nom du solveur ne suffit pas a savoir ce qu'il oublie.
# Famille : Brown & Sandholm 2019 (Discounted CFR).
class DCFRDiscountedSolver(CFRSolver):
    """DCFR Discounted : combinaison de CFR+ (clip) + discount des regrets
    anciens. C'est le 'DCFR' reference dans le papier OFTRL Farina 2019."""

    def __init__(self, alpha: float = 1.5, beta: float = 0.0, gamma: float = 2.0):
        super().__init__()
        self._alpha = alpha  # discount sur regret_sum
        self._beta = beta    # discount sur strategy_sum (si > 0)
        self._gamma = gamma  # clipping CFR+ sur les regrets negatifs

    def cfr(self, history, cards, reach_probs):
        if self.game.is_terminal(history):
            payoff = self.game.get_payoff(history, cards)
            return np.array([payoff, -payoff])
        player = self.game.get_current_player(history)
        info_set = self.game.get_info_set(history, cards[player])
        strategy = self.get_strategy(info_set)
        action_utils = np.zeros(self.game.NUM_ACTIONS)
        node_value = 0.0
        for a in range(self.game.NUM_ACTIONS):
            new_history = history + ("p" if a == 0 else "b")
            child_value = self.cfr(new_history, cards,
                                   reach_probs * strategy[a])
            action_utils[a] = child_value[player]
            node_value += strategy[a] * action_utils[a]
        regrets = action_utils - node_value
        # CFR+ clipping : max avec 0 + ponderation
        positive_regrets = np.maximum(regrets, 0)
        t = max(self.iterations, 1)
        # Discount factor = t^alpha / (t^alpha + 1)
        discount = (t ** self._alpha) / (t ** self._alpha + 1)
        self.regret_sum[info_set] += reach_probs[1 - player] * positive_regrets * discount
        # Cumulative strategy discount
        strat_discount = (t ** self._beta) / (t ** self._beta + 1) if self._beta > 0 else 1.0
        self.strategy_sum[info_set] += reach_probs[0] * strategy * strat_discount
        return np.array([node_value, -node_value]) if player == 0 else np.array(
            [-node_value, node_value]
        )


# Sanity check : convergence plus rapide sur l info-set Qp (J1 carte Q face a un check --
# info-set effectivement visite par la recursion avec cards=[0,1] ; l info-set Q = carte Q
# a la racine n existe pas ici car le joueur 0 agit a la racine, cf issue #21488)
#
# Les quatre solveurs sont instancies NEUFS ici. Reutiliser ceux de la cellule 4
# laisserait 500 mises a jour dans leurs regrets alors que leur compteur repart a 0 :
# le point "t=10" comparerait alors 510 mises a jour a 10, et la these du commentaire
# (DCFR converge plus vite) serait evaluee sur des compteurs inegaux.
cfr = CFRSolver()
cfr_plus = CFRPlusSolver()
oftrl = OFTRLSolver(kappa=1.0, alpha=0.5, beta=1.0)
cfr_d = DCFRDiscountedSolver()
print("DCFR sanity check:")
for t in (10, 100, 500):
    while cfr.iterations < t:
        cfr.iterations += 1
        cfr.cfr("", [0, 1], np.array([1.0, 1.0]))
        cfr_plus.iterations = cfr.iterations
        cfr_plus.cfr("", [0, 1], np.array([1.0, 1.0]))
        oftrl.iterations = cfr.iterations
        oftrl.cfr("", [0, 1], np.array([1.0, 1.0]))
        cfr_d.iterations = cfr.iterations
        cfr_d.cfr("", [0, 1], np.array([1.0, 1.0]))
    print(f"  t={t:4d}  CFR={cfr.get_average_strategy('Qp').round(3).tolist()}  "
          f"DCFR={cfr_d.get_average_strategy('Qp').round(3).tolist()}")


DCFR sanity check:
  t=  10  CFR=[0.05, 0.95]  DCFR=[0.034, 0.966]
  t= 100  CFR=[0.005, 0.995]  DCFR=[0.003, 0.997]


  t= 500  CFR=[0.001, 0.999]  DCFR=[0.001, 0.999]


### 1.5 L'instrument de mesure : la meilleure réponse, pas une distance à l'uniforme

Une comparaison honnête exige de neutraliser tout sauf l'opérateur de mise à jour : **même budget d'itérations**, **mêmes donnes**, **mêmes adversaires**. La cellule suivante instancie les quatre variantes — CFR, CFR+, DCFR, OFTRL — sur Kuhn poker et rapporte deux mesures : le **temps de calcul** (`perf_counter`) et l'**exploitabilité réelle**, c'est-à-dire ce qu'un meilleur-respondant optimal peut encore gagner contre la stratégie moyenne (`get_exploitability()` du socle, qui délègue le calcul de meilleure réponse à OpenSpiel). Zéro = équilibre de Nash.

Deux précautions que ce carnet a dû corriger :

- **la distance à la stratégie uniforme n'est pas une exploitabilité.** Elle mesure l'éloignement de $[0{,}5, 0{,}5]$, pas l'éloignement de Nash — et sur l'info-set `Q` ces deux distances se contredisent : la stratégie de Nash y est $[1, 0]$ (ne jamais miser), donc une stratégie à $[0{,}9995, 0{,}0005]$ est **quasi parfaite** tout en étant très loin de l'uniforme. Classer les solveurs sur cette distance retourne le verdict ;
- **les quatre solveurs sont déterministes.** Aucun d'eux ne tire de nombre au hasard : `np.random.seed` ne les atteint pas. Un axe « 4 graines » produirait huit lignes identiques à l'octet et ne mesurerait aucune variation — la cellule le vérifie et le dit, au lieu de le suggérer.

Les donnes sont énumérées **en entier** (six donnes, pas trois) : avec trois donnes seulement, les info-sets où le joueur 2 détient un roi ne sont jamais visités, gardent leur stratégie uniforme initiale, et l'exploitabilité plafonne quel que soit le budget. La cellule suivante mesure cet effet au lieu de le supposer.


In [4]:
# Benchmark Kuhn Poker : exploitabilite REELLE (meilleure reponse) + temps de calcul
import time

# References theoriques sous les payoffs de ce carnet (KuhnPoker du socle) :
# la valeur de jeu du joueur 0 vaut -1/18, et la strategie de Nash au noeud 'Q'
# (joueur 0 detenant une dame, a la racine) est de ne JAMAIS miser.
VALEUR_JEU_J0 = -1.0 / 18.0
NASH_Q = np.array([1.0, 0.0])

# Toutes les donnes. Trois donnes seulement laissaient les info-sets ou le joueur 2
# detient un roi (Kp, Kb) sans aucune mise a jour : ils restaient a [0.5, 0.5] et
# plafonnaient l'exploitabilite, quel que soit le budget.
DONNES = [(a, b) for a in range(3) for b in range(3) if a != b]


def construire_solveurs():
    """Les quatre solveurs, instancies NEUFS (leurs regrets ne doivent pas etre partages)."""
    return [
        ("CFR", CFRSolver()),
        ("CFR+", CFRPlusSolver()),
        ("DCFR", DCFRDiscountedSolver()),
        ("OFTRL", OFTRLSolver()),
    ]


def entrainer(solveurs, iterations, donnes_par_iteration):
    """Meme budget d'iterations pour les quatre ; `donnes_par_iteration(t)` choisit les donnes."""
    for t in range(iterations):
        for cp in donnes_par_iteration(t):
            for _nom, solveur in solveurs:
                solveur.iterations = t + 1
                solveur.cfr("", list(cp), np.array([1.0, 1.0]))


def mesurer(iterations, mode):
    """Un bras du benchmark. Aucune graine : les solveurs sont deterministes."""
    solveurs = construire_solveurs()
    if mode == "simultaneous":
        # chaque iteration parcourt le jeu entier
        plan = lambda t: DONNES
    else:
        # chaque iteration ne parcourt qu'une donne sur six, en rotation :
        # meme nombre d'iterations, trois fois moins de mises a jour
        plan = lambda t: [DONNES[t % len(DONNES)]]
    debut = time.perf_counter()
    entrainer(solveurs, iterations, plan)
    ecoule = time.perf_counter() - debut
    return {
        "mode": mode,
        "secondes": round(ecoule, 3),
        "lignes": [
            {
                "solveur": nom,
                "exploitabilite": float(solveur.get_exploitability()),
                "Q": solveur.get_average_strategy("Q").round(4).tolist(),
            }
            for nom, solveur in solveurs
        ],
    }


BUDGET = 1000
resultats = [mesurer(BUDGET, "simultaneous"), mesurer(BUDGET, "alternating")]

print(f"Kuhn poker, {BUDGET} iterations, six donnes.")
print(f"Reference : valeur de jeu du joueur 0 = {VALEUR_JEU_J0:+.6f} ; Nash sur 'Q' = {NASH_Q.tolist()}")
print()
print(f"{'mode':>13} {'solveur':>7} {'exploitabilite':>15} {'strategie Q':>22} {'temps total':>12}")
for bras in resultats:
    for ligne in bras["lignes"]:
        print(f"{bras['mode']:>13} {ligne['solveur']:>7} {ligne['exploitabilite']:>15.6f} "
              f"{str(ligne['Q']):>22} {bras['secondes']:>11.3f}s")

print()
print("Classement par exploitabilite (plus petit = plus proche de Nash) :")
for bras in resultats:
    ordre = sorted(bras["lignes"], key=lambda l: l["exploitabilite"])
    print(f"  {bras['mode']:>13} : " + " < ".join(f"{l['solveur']} ({l['exploitabilite']:.4f})" for l in ordre))

# Controle de determinisme : deux tirages a graines differentes donnent-ils la meme strategie ?
# (Si oui, un axe de graines ne mesurerait rien.)
strategies = []
for graine in (0, 42):
    np.random.seed(graine)
    solveurs = construire_solveurs()
    entrainer(solveurs, 300, lambda t: DONNES)
    strategies.append([s.get_average_strategy("Q").round(6).tolist() for _n, s in solveurs])
print()
print(f"Controle de determinisme (300 iterations, graines 0 et 42) : "
      f"strategies identiques = {strategies[0] == strategies[1]}")

# Effet du jeu de donnes, mesure au lieu d'etre affirme
trois_donnes = [(0, 1), (1, 0), (2, 1)]
for etiquette, donnes in (("3 donnes (ancienne boucle)", trois_donnes), ("6 donnes", DONNES)):
    solveurs = construire_solveurs()
    entrainer(solveurs, BUDGET, lambda t, d=donnes: d)
    valeurs = {nom: round(float(s.get_exploitability()), 6) for nom, s in solveurs}
    print(f"  {etiquette:26s} CFR={valeurs['CFR']:.6f}  CFR+={valeurs['CFR+']:.6f}")


Kuhn poker, 1000 iterations, six donnes.
Reference : valeur de jeu du joueur 0 = -0.055556 ; Nash sur 'Q' = [1.0, 0.0]

         mode solveur  exploitabilite            strategie Q  temps total
 simultaneous     CFR        0.006505       [0.9988, 0.0012]       4.310s
 simultaneous    CFR+        0.006784       [0.9889, 0.0111]       4.310s
 simultaneous    DCFR        0.071657       [0.6208, 0.3792]       4.310s
 simultaneous   OFTRL        0.018525       [0.9986, 0.0014]       4.310s
  alternating     CFR        0.018506       [0.9929, 0.0071]       0.687s
  alternating    CFR+        0.019493       [0.9655, 0.0345]       0.687s
  alternating    DCFR        0.073547       [0.6184, 0.3816]       0.687s
  alternating   OFTRL        0.022055       [0.9924, 0.0076]       0.687s

Classement par exploitabilite (plus petit = plus proche de Nash) :
   simultaneous : CFR (0.0065) < CFR+ (0.0068) < OFTRL (0.0185) < DCFR (0.0717)
    alternating : CFR (0.0185) < CFR+ (0.0195) < OFTRL (0.0221) < 


Controle de determinisme (300 iterations, graines 0 et 42) : strategies identiques = True


  3 donnes (ancienne boucle) CFR=0.193655  CFR+=0.193131


  6 donnes                   CFR=0.006505  CFR+=0.006784


## 2. Verdict honnête

Le benchmark précédent **retourne** la lecture attendue au §1.3 : à budget d'itérations égal, sur Kuhn poker, le **DCFR est dernier** — dans les deux bras, et de loin.

| Solveur | Exploitabilité, `simultaneous` | `alternating` | Stratégie apprise sur `Q` (Nash : `[1, 0]`) |
|---|---|---|---|
| CFR | **0,006505** | 0,018506 | `[0,9988, 0,0012]` |
| CFR+ | 0,006784 | 0,019493 | `[0,9889, 0,0111]` |
| OFTRL | 0,018525 | 0,022055 | `[0,9986, 0,0014]` |
| DCFR | 0,071657 | 0,073547 | `[0,6208, 0,3792]` |

Ce que ces quatre lignes disent, et ce qu'elles ne disent pas :

- **DCFR est onze fois plus exploitable que CFR** (0,0717 contre 0,0065) et sa stratégie sur `Q` mise **38 % du temps avec une dame**, là où l'équilibre ne mise jamais. Ce n'est pas une nuance de classement, c'est un solveur qui n'a pas convergé.
- **CFR+ ne devance pas CFR** ici (0,0068 contre 0,0065) : à ce budget, l'écart entre les deux est plus petit que ce que le banc peut distinguer. Annoncer une hiérarchie serait surinterpréter.
- **OFTRL non réglé est intermédiaire** (0,0185) : le pas optimiste coûte quand ses paramètres ne sont pas ajustés au jeu, ce qui est exactement la mise en garde des auteurs — mais il reste **meilleur que le DCFR** sur ce banc.
- **Le bras `alternating` ne fait pas ralentir les solveurs : il fait moins de travail.** Il parcourt une donne par itération au lieu de six, d'où 0,69 s contre 4,31 s, et une exploitabilité plus haute à nombre d'itérations égal. L'ancienne prose y lisait « l'avantage optimiste effacé » : il n'y a rien à effacer, seulement six fois moins de mises à jour.

**Ce que ce verdict ne prétend pas.** Il mesure **une implémentation** : celle du §1.4, dont les facteurs portent sur l'incrément de regret et ne décroissent pas la somme accumulée. Une implémentation de référence — où l'ancien est effectivement oublié — n'est pas mesurée ici, et rien dans ce carnet ne dit que le DCFR serait mauvais en général. C'est la distinction que le carnet enseigne désormais : **la réputation d'un algorithme ne se transporte pas d'un code à l'autre**, et « l'article l'affirme » n'est pas « je le mesure ici ».


## 3. Exercices

### Exercice 1 — Régler $(\kappa, \alpha, \beta)$ par profondeur

**Énoncé** : Soit le treeplex de Kuhn Poker de profondeur 2 (cartes → action → action). Ajuster $(\kappa, \alpha, \beta)$ pour maximiser la convergence de l'OFTRL en 100 itérations sur le mode `simultaneous`. Mesurer la vitesse avec l'exploitabilité réelle (`get_exploitability()`, meilleure réponse), et non avec une distance à l'uniforme qui classe mal les stratégies.

**Indices** :
- `depth` est passé récursivement à `cfr()` ; il borne la profondeur de l'arbre
- $(\kappa, \alpha, \beta)$ trop petits → le pas optimiste disparaît, on retombe sur CFR
- $(\kappa, \alpha, \beta)$ trop grands → oscillations, le regret oscille au lieu de converger
- Pour Kuhn (profondeur 2), les auteurs recommandent typiquement $\kappa \approx 1.5$, $\alpha \approx 0.3$, $\beta \approx 0.5$


In [5]:
# Exercice 1 — squelette a completer
def exercise_1_oftl_tuning():
    """Trouver (kappa, alpha, beta) qui maximise la convergence OFTRL en 100 iter
    sur Kuhn Poker simultane. Retourner la distance L2 a la strategie uniforme
    sur l'info-set 'Q'."""
    best = None
    best_distance = float("inf")
    # TODO etudiant : grid search sur (kappa, alpha, beta)
    # pour chaque combinaison, executer 100 iterations et mesurer
    # la distance a la strategie Nash theorique sur 'Q'.
    return best, best_distance

print("Exercice 1 squelette pret. A completer par l'etudiant.")
print("Indice : utiliser une grid search externe sur itertools.product,")
print("mesurer distance_to_uniform apres 100 iter, retourner le meilleur (kappa, alpha, beta).")


Exercice 1 squelette pret. A completer par l'etudiant.
Indice : utiliser une grid search externe sur itertools.product,
mesurer distance_to_uniform apres 100 iter, retourner le meilleur (kappa, alpha, beta).


### Exercice 2 — Modes simultanés vs alternés sur DCFR

**Énoncé** : Comparer le comportement de DCFR dans les deux bras du benchmark — `simultaneous` (chaque itération parcourt les six donnes) et `alternating` (chaque itération ne parcourt qu'une donne, en rotation, donc trois fois moins de mises à jour). Tracer la courbe d'exploitabilité réelle (`get_exploitability()`) en fonction des itérations, pour 1000 itérations. Les solveurs étant déterministes, il n'y a pas de graine à faire varier : ce qui varie est le nombre de mises à jour par itération.

**Indices** :
- DCFR avec alternation tend à être **plus stable** car chaque joueur voit moins de bruit adversarial
- mais DCFR simultané converge **plus vite** en moyenne sur petits jeux
- Vérifier empiriquement sur les 4 seeds si l'effet est significatif


In [6]:
# Exercice 2 — tracer NashConv iteration par iteration
def exercise_2_dcfr_alternating():
    """Pour les 2 bras du benchmark, mesurer l'exploitabilite reelle (get_exploitability())
    tous les 50 iterations sur 1000 iterations DCFR. Retourner un dict mode -> [(iter, exploitabilite), ...].
    Les solveurs sont deterministes : il n'y a pas de graine a faire varier."""
    # TODO etudiant : remplir la structure result[mode] = [(iter, exploitabilite), ...]
    return {}

print("Exercice 2 squelette pret. A completer par l'etudiant.")
print("Indice : utiliser plt.plot sur les resultats, ajouter label (seed, mode).")


Exercice 2 squelette pret. A completer par l'etudiant.
Indice : utiliser plt.plot sur les resultats, ajouter label (seed, mode).


### Exercice 3 — Prédiction contrefactuelle sur petit jeu synthétique

**Énoncé** : Créer un jeu à 2 joueurs à information imparfaite avec 3 actions et 4 informations sets. Implémenter `OFCFPredictor` (version simplifiée de l'algorithme 2 de Farina et al.) qui produit la **prédiction** $m_{t+1}$ à partir des 5 dernières observations de regret. Comparer la convergence à CFR vanilla sur ce jeu.

**Indices** :
- Définir la classe `MiniGame3x4(NUM_ACTIONS=3, NUM_INFO_SETS=4)` avec `get_payoff(history, cards)`
- $m_{t+1}$ = moyenne des 5 derniers regrets positifs
- Une bonne prédiction devrait accélérer la convergence mesurée par $R^T$


In [7]:
# Exercice 3 — Mini-jeu + OFCFPredictor
class MiniGame3x4:
    """Mini-jeu synthetique 3 actions, 4 info sets pour exercice 3."""
    NUM_ACTIONS = 3
    NUM_INFO_SETS = 4

    def __init__(self):
        np.random.seed(0)
        # Payoffs aleatoires par info-set (fixed seed pour reproductibilite)
        self.payoffs = np.random.uniform(-1, 1, (4, 3))

    def get_payoff(self, history: str, card: int) -> np.ndarray:
        info_set = hash(history + str(card)) % self.NUM_INFO_SETS
        return self.payoffs[info_set]

    def is_terminal(self, history: str) -> bool:
        return len(history) >= 2

    def get_current_player(self, history: str) -> int:
        return len(history) % 2

    def get_info_set(self, history: str, card: int) -> str:
        return f"I{hash(history+str(card)) % self.NUM_INFO_SETS}"

    def get_actions(self, history: str) -> List[int]:
        return [0, 1, 2]


# Stub a completer par l'etudiant
class OFCFPredictor:
    """OFCF = Optimistic Counterfactual Factor. Predire le regret a t+1 par
    moyenne des 5 derniers regrets positifs."""

    def __init__(self, window: int = 5):
        self.window = window
        self.history = defaultdict(list)

    def predict(self, info_set: str, current_regrets: np.ndarray) -> np.ndarray:
        # TODO etudiant : retourner moyenne des self.window derniers regrets positifs
        # pour info_set, ou np.zeros_like(current_regrets) si historique vide.
        return np.zeros_like(current_regrets)


print("Exercice 3 squelette pret. Mini-jeu + predict stub.")


Exercice 3 squelette pret. Mini-jeu + predict stub.


## Conclusion

Ce notebook compagnon de GT-13 illustre la **frontière théorie/pratique** du CFR optimiste — à condition de mesurer au lieu de réciter :

| Solveur | Garantie de regret | Exploitabilité mesurée ici (§2) |
|---|---|---|
| CFR | $\mathcal{O}(\sqrt{T})$ | la plus basse du banc |
| CFR+ | $\mathcal{O}(\sqrt{T})$, constantes réduites | indiscernable de CFR à ce budget |
| DCFR | $\mathcal{O}(\sqrt{T})$ empiriquement | **la plus haute du banc** |
| OFTRL (paramètres par défaut) | $\mathcal{O}(T^{-3/4})$ | intermédiaire |
| OFTRL réglé empiriquement | (idem) | non mesuré ici — c'est l'exercice 1 |

**Leçon** : la supériorité théorique de l'OFTRL sur grand treeplex (Libratus-scale) ne se traduit pas automatiquement en supériorité empirique sur petit jeu, **et la réputation empirique du DCFR ne suit pas une implémentation qui ne fait pas ce que son nom annonce**. Les deux écarts se mesurent ; aucun ne se récite.

**Sources complémentaires** :
- Farina, Kroer, Brown, Sandholm, *Stable-Predictive Optimistic Counterfactual Regret Minimization*, PMLR 97 (2019)
- Brown, Sandholm, *Solving Imperfect-Information Games via Discounted Regret Minimization* (DCFR référence)
- Farina, Kroer, Sandholm, *Regret Circuits: Composability of Regret Minimizers*, ICML 2019 (socle #13350)

Voir aussi : `#13291` (issue dispatcheur), `#13290` (dépendance socle), `#13350` (socle mergé).
